# PSR regression quality per joint and LOTO fold (Table 3, Supp S1)

Fits PSR weights per fold (strict LOTO), computes per-joint R-squared and RMSE. Produces Table 3 and Supplementary Table S1.


**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).

**Input dependency.** This notebook is a T4-row extension. It reads the existing 3-fold AUROC values from a CSV produced by an earlier run, computes the T4 row, and merges. Required input:

- `Processed_Data/NB10d_regression_quality.csv`

If the file is missing, the notebook raises an explicit error.


In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
import h5py

warnings.filterwarnings("ignore")
np.random.seed(42)

ROOT = r"D:\Research\RCM"
BASE = os.path.join(ROOT, "Lab_Data")
OUT  = os.path.join(ROOT, "Processed_Data")

CANONICAL_CSV = os.path.join(OUT, "NB10d_regression_quality.csv")
OUT_CSV       = os.path.join(OUT, "NB10d_regression_quality_4fold.csv")

# Constants
TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}
PAYLOAD_COM  = np.array([0, 0, 0.05])
GRAVITY      = np.array([0, 0, -9.81])
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200

UR5_DH_A     = [0, -0.42500, -0.39225, 0, 0, 0]
UR5_DH_D     = [0.089159, 0, 0, 0.10915, 0.09465, 0.0823]
UR5_DH_ALPHA = [np.pi/2, 0, 0, np.pi/2, -np.pi/2, 0]
UR5_MASS     = [3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879]
UR5_COM      = [
    [0.0,     -0.02561,  0.00193],
    [0.21250,  0.0,      0.11336],
    [0.11993,  0.0,      0.02650],
    [0.0,     -0.00180,  0.01634],
    [0.0,      0.00180,  0.01634],
    [0.0,      0.0,     -0.00116],
]

def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,    sa,    ca,    d   ],
        [0,    0,     0,     1   ]
    ])

def gravity_torque(q, payload_mass=0.0):
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(
            UR5_DH_A[i], UR5_DH_D[i], UR5_DH_ALPHA[i], q[i]))
    com_world = [(T[i+1] @ np.array([*UR5_COM[i], 1.0]))[:3]
                 for i in range(6)]
    masses = list(UR5_MASS)
    if payload_mass > 0:
        masses.append(payload_mass)
        com_world.append((T[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        qp = q.copy(); qp[i] += dq
        Tp = [np.eye(4)]
        for jj in range(6):
            Tp.append(Tp[-1] @ dh_transform(
                UR5_DH_A[jj], UR5_DH_D[jj], UR5_DH_ALPHA[jj], qp[jj]))
        for jj in range(len(masses)):
            cp = (Tp[jj+1] @ np.array([*UR5_COM[jj], 1.0]))[:3] if jj < 6 \
                 else (Tp[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3]
            tau_g[i] += masses[jj] * GRAVITY @ ((cp - com_world[jj]) / dq)
    return tau_g

# REGISTRY (T1/T2/T3 + T4)
REGISTRY = {
    "T1_healthy":    ("T1_PickPlace/Healthy",  "UR5_T1_healthy_180cyc_*.h5",        "T1"),
    "T2_healthy":    ("T2_Assembly/Healthy",   "UR5_T2_healthy_180cyc_*.h5",        "T2"),
    "T3_healthy":    ("T3_Palletize/Healthy",  "UR5_T3_healthy_183cyc_*.h5",        "T3"),
    "T4_healthy":    ("T4_BinReorient/healthy","UR5_T4_healthy_session2_35cyc_*.h5","T4"),
}

def load_healthy_cycles(task_filter=None):
    """Return list of cycle dicts (q, qd, current, task)."""
    cycles = []
    for key, (subdir, pattern, task) in REGISTRY.items():
        if task_filter is not None and task not in task_filter:
            continue
        matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
        if not matches:
            print(f"  WARNING  Not found: {key}")
            continue
        with h5py.File(matches[0], "r") as f:
            cnum    = f["cycle_number"][:].astype(int).ravel()
            cur_all = f["actual_current"][:]
            q_all   = f["actual_q"][:]
            qd_all  = f["actual_qd"][:]
        for c in np.unique(cnum[cnum > 0]):
            mask = cnum == c
            if mask.sum() >= MIN_SAMP:
                cycles.append({
                    "q": q_all[mask], "qd": qd_all[mask],
                    "current": cur_all[mask], "task": task,
                })
    return cycles

# Build Phi and I for the OLS regression
def build_phi_I(cycles):
    """Build per-joint (Phi_j, I_j) arrays across all cycles."""
    Phi_per_joint = {j: [] for j in range(6)}
    I_per_joint   = {j: [] for j in range(6)}
    for cyc in cycles:
        payload = TASK_PAYLOAD[cyc["task"]]
        q_a = cyc["q"]; qd_a = cyc["qd"]; cur = cyc["current"]
        N   = len(q_a)
        for t in range(0, N, SUBSAMPLE):
            tau_g = gravity_torque(q_a[t], payload_mass=payload)
            for j in range(6):
                if 0 < t < N - 1:
                    qdd_j = (qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                else:
                    qdd_j = 0.0
                Phi_per_joint[j].append(
                    np.array([tau_g[j], qd_a[t,j],
                              np.sign(qd_a[t,j]), qdd_j, 1.0]))
                I_per_joint[j].append(cur[t, j])
    return ({j: np.array(Phi_per_joint[j]) for j in range(6)},
            {j: np.array(I_per_joint[j])   for j in range(6)})

def fit_psr_weights(Phi_train, I_train):
    """Plain OLS per joint, identical to cell 24."""
    W = {}
    for j in range(6):
        w, _, _, _ = np.linalg.lstsq(Phi_train[j], I_train[j], rcond=None)
        W[j] = w
    return W

# Main
print("=" * 70)
print("NB10d_T4_extension -- compute T4 row for regression_quality CSV")
print("=" * 70)

print(f"\n[Step 1] Loading canonical {os.path.basename(CANONICAL_CSV)}...")
if not os.path.exists(CANONICAL_CSV):
    raise RuntimeError(
        f"Canonical regression-quality CSV not found at {CANONICAL_CSV}. "
        "This file is generated by the original NB10d (cell 24) and is required "
        "as the source of the published Table 2 / Supp S1 values."
    )
canon = pd.read_csv(CANONICAL_CSV)
print(f"  Loaded {len(canon)} canonical rows (T1, T2, T3 folds x 6 joints).")
print(f"  Tasks present: {sorted(canon['test_task'].unique())}")

# Step 2: load all healthy cycles
print("\n[Step 2] Loading healthy cycles (T1+T2+T3 for training; T4 for test)...")
all_healthy = load_healthy_cycles(task_filter=["T1", "T2", "T3", "T4"])
tr_cycles = [c for c in all_healthy if c["task"] in ["T1", "T2", "T3"]]
te_cycles = [c for c in all_healthy if c["task"] == "T4"]
print(f"  Training cycles (T1+T2+T3): {len(tr_cycles)}")
print(f"  Test cycles (T4): {len(te_cycles)}")

# Step 3: fit PSR on training pool
print("\n[Step 3] Fitting PSR weights on T1+T2+T3 healthy...")
t0 = time.perf_counter()
Phi_tr, I_tr = build_phi_I(tr_cycles)
W = fit_psr_weights(Phi_tr, I_tr)
print(f"  Fit time: {time.perf_counter()-t0:.1f} s")

# Step 4: evaluate per-joint R^2 on T4
print("\n[Step 4] Evaluating per-joint R²/RMSE on T4 healthy...")
Phi_te, I_te = build_phi_I(te_cycles)

t4_rows = []
print(f"\n  Fold: test=T4  train tasks=['T1', 'T2', 'T3']  "
      f"train_n={len(tr_cycles)}  test_n={len(te_cycles)}")
for j in range(6):
    I_pred  = Phi_te[j] @ W[j]
    I_meas  = I_te[j]
    resid   = I_meas - I_pred
    ss_res  = np.sum(resid**2)
    ss_tot  = np.sum((I_meas - I_meas.mean())**2)
    r2      = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse    = np.sqrt(np.mean(resid**2))
    mae     = np.mean(np.abs(resid))
    bias    = resid.mean()
    res_std = resid.std()
    print(f"    J{j+1}: R²={r2:+.4f}  RMSE={rmse:.4f} A  "
          f"MAE={mae:.4f} A  bias={bias:+.4f} A  std={res_std:.4f} A")
    t4_rows.append(dict(
        test_task="T4",
        train_tasks="T1+T2+T3",
        joint=j,
        joint_label=f"J{j+1}",
        n_train_points=len(Phi_te[j]),
        n_train_cycles=len(te_cycles),
        R2=round(float(r2), 5),
        RMSE_A=round(float(rmse), 5),
        MAE_A=round(float(mae), 5),
        bias_A=round(float(bias), 5),
        resid_std_A=round(float(res_std), 5),
    ))

# Step 5: combine and save
print("\n[Step 5] Merging T4 rows with canonical CSV...")
combined = pd.concat([canon, pd.DataFrame(t4_rows)], ignore_index=True)
combined.to_csv(OUT_CSV, index=False)
print(f"  Saved: {OUT_CSV}")
print(f"  Total rows: {len(combined)} ({len(canon)} canonical + {len(t4_rows)} new T4)")

# Step 6: print Table 2 (R²min summary)
print("\n" + "=" * 70)
print("REVISED TABLE 2 (R²min diagnostic, 4 folds)")
print("=" * 70)
print(f"  {'Test fold':<12}{'Training':<14}{'n train':<10}{'R²min':<10}"
      f"{'Worst joint':<14}{'Mean R²':<10}")
print("  " + "-" * 70)

# Map test_task -> training pool size (from cell 24 conventions)
# T1: T2+T3 healthy; T2: T1+T3; T3: T1+T2; T4: T1+T2+T3
TRAIN_N_MAP = {
    "T1": (363, ["T2", "T3"]),
    "T2": (363, ["T1", "T3"]),
    "T3": (360, ["T1", "T2"]),
    "T4": (len(tr_cycles), ["T1", "T2", "T3"]),
}

for task in ["T1", "T2", "T3", "T4"]:
    sub = combined[combined["test_task"] == task]
    r2_min  = sub["R2"].min()
    worst   = sub.loc[sub["R2"].idxmin(), "joint_label"]
    r2_mean = sub["R2"].mean()
    n_tr, tr_list = TRAIN_N_MAP[task]
    print(f"  {task:<12}{'+'.join(tr_list):<14}{n_tr:<10}"
          f"{r2_min:+.3f}    {worst:<14}{r2_mean:.3f}")
print("=" * 70)

# Step 7: print Supp S1 (per-joint R² and RMSE across folds)
print("\nREVISED SUPP TABLE S1 (per-joint R²/RMSE across 4 folds)")
print("=" * 95)
print(f"  {'Joint':<8}" +
      "".join([f"{f'{t} R²':<10}{f'{t} RMSE(A)':<14}" for t in ["T1","T2","T3","T4"]]))
print("  " + "-" * 95)
for j in range(6):
    row = f"  {'J'+str(j+1):<8}"
    for t in ["T1", "T2", "T3", "T4"]:
        sub = combined[(combined["test_task"] == t) & (combined["joint"] == j)]
        if len(sub) == 0:
            row += f"{'—':<10}{'—':<14}"
            continue
        r2   = sub.iloc[0]["R2"]
        rmse = sub.iloc[0]["RMSE_A"]
        marker = "‡" if r2 < 0 else ""
        row += f"{r2:+.3f}{marker:<4}{rmse:.3f}         "
    print(row)
print("=" * 95)
print("‡ negative R² indicates the PSR model fails to outperform the mean predictor")

# Step 8: sanity check
print("\n[Integrity check] T1/T2/T3 R² preserved byte-for-byte:")
match = True
for _, canon_row in canon.iterrows():
    merged_row = combined[(combined["test_task"] == canon_row["test_task"]) &
                          (combined["joint"] == canon_row["joint"])]
    if len(merged_row) != 1 or float(merged_row.iloc[0]["R2"]) != float(canon_row["R2"]):
        match = False
        print(f"  DRIFT: {canon_row['test_task']} J{canon_row['joint']+1} "
              f"canonical={canon_row['R2']} merged={merged_row.iloc[0]['R2']}")
if match:
    print(f"  All {len(canon)} T1/T2/T3 rows match canonical CSV exactly.")

print("\nNB10d_T4_extension complete.")